# INNOBEST — Ejecutor de entrenamientos en GPU T4
Este cuaderno ejecuta los trabajos que se solicitan **desde Administración de IA**. Mantén abierta la sesión Colab para completar las 100 épocas de cada modelo. En Render no hay GPU T4 incluida.
**Importante:** no subas archivos `.env` ni compartas la URI de MongoDB. Autoriza temporalmente la IP pública de Colab en Atlas si la conexión está bloqueada.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Activa Entorno de ejecución → Cambiar tipo de entorno → T4 GPU"


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
print("Sube el ZIP INNOBEST con el worker GPU (sin .env)")
archivos = files.upload()
nombre = next(iter(archivos))
with ZipFile(nombre) as z:
    for item in z.infolist():
        ruta = Path(item.filename)
        if ruta.is_absolute() or ".." in ruta.parts:
            raise ValueError("ZIP con rutas inseguras")
    z.extractall("/content")
carpetas = [p for p in Path("/content").glob("*/scripts/worker_gpu.py")]
assert len(carpetas)==1, "No se encontró scripts/worker_gpu.py en el ZIP"
proyecto = carpetas[0].parents[1]
print("Código listo:", proyecto)


In [ ]:
import os, sys, subprocess
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(proyecto/"requirements.txt")],check=True)
from getpass import getpass
os.environ["MONGODB_URI"] = getpass("Introduce MONGODB_URI completa (no se mostrará): ")
os.environ["MONGODB_DB"] = "innobest_huanuco"
from pymongo import MongoClient
MongoClient(os.environ["MONGODB_URI"],serverSelectionTimeoutMS=8000).admin.command("ping")
print("MongoDB conectado. No compartas credenciales.")


## Ejecutar trabajos solicitados desde la web
Esta celda queda funcionando, consulta la cola cada 15 segundos y entrena con la GPU de Colab. **No cierres ni desconectes Colab durante un entrenamiento.** Los modelos resultantes se guardan en MongoDB para descargarlos desde la web.

In [ ]:
import subprocess, sys
ejecutor = subprocess.Popen([sys.executable,"-u",str(proyecto/"scripts/worker_gpu.py")], cwd=str(proyecto),env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for linea in ejecutor.stdout:
    print(linea,end="",flush=True)
